![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Turning one flat file into a relational database: clean the data, split it into tables, wire up the keys, load it into SQLite.

> **Done when:** `data/project.db` exists with 3+ related tables, `sql/schema.sql` matches your ERD, no foreign key is dangling, and the row counts are what you expect.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [23]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

In [109]:
demo = pd.read_sas("../data/raw/DEMO_L.XPT", format="xport")
activity = pd.read_sas("../data/raw/PAQ_L.XPT", format="xport")
body = pd.read_sas("../data/raw/BMX_L.XPT", format="xport")
bp = pd.read_sas("../data/raw/BPXO_L.XPT", format="xport")

---
## 2. Clean

Work through the problems you listed in notebook 01. Every decision you make here — drop the nulls or keep them, cast or discard — is one you may be asked to defend on Friday, so note why.

In [110]:
#First renaming the column to undestand the data better
demo = demo.rename(columns={
    "SEQN": "participant_id",
    "SDDSRVYR": "survey_cycle",
    "RIDSTATR": "interview_exam_status",
    "RIAGENDR": "sex",
    "RIDAGEYR": "age_years",
    "RIDAGEMN": "age_months_screening",
    "RIDRETH1": "race_ethnicity",
    "RIDRETH3": "race_ethnicity_asian_detail",
    "RIDEXMON": "exam_period",
    "RIDEXAGM": "age_months_exam",
    "DMQMILIZ": "military_service",
    "DMDBORN4": "country_of_birth",
    "DMDYRUSR": "years_in_us",
    "DMDEDUC2": "education_level",
    "DMDMARTZ": "marital_status",
    "RIDEXPRG": "pregnancy_status",
    "DMDHHSIZ": "household_size",
    "DMDHRGND": "household_reference_sex",
    "DMDHRAGZ": "household_reference_age",
    "DMDHREDZ": "household_reference_education",
    "DMDHRMAZ": "household_reference_marital_status",
    "DMDHSEDZ": "household_spouse_education",
    "WTINT2YR": "interview_weight",
    "WTMEC2YR": "exam_weight",
    "SDMVSTRA": "survey_stratum",
    "SDMVPSU": "survey_psu",
    "INDFMPIR": "income_poverty_ratio"
})
#Extracting the columns that I think is important to respond the scientific questions.
demo = demo[
    [
        "participant_id",
        "age_years",
        "sex",
        "exam_weight",
        "education_level",
        "income_poverty_ratio"
    ]
]
demo

,participant_id,age_years,sex,exam_weight,education_level,income_poverty_ratio
0,130378.0,43.0,1.0,54374.463898,5.0,5.00
1,130379.0,66.0,1.0,34084.721548,5.0,5.00
2,130380.0,44.0,2.0,81196.277992,3.0,1.41
3,130381.0,5.0,2.0,55698.607106,NaN,1.53
4,130382.0,2.0,1.0,36434.146346,NaN,3.60
...,...,...,...,...,...,...
11928,142306.0,9.0,1.0,13459.129019,NaN,2.01
11929,142307.0,49.0,2.0,64962.328962,5.0,NaN
11930,142308.0,50.0,1.0,44367.534132,4.0,1.95
11931,142309.0,40.0,1.0,46249.361849,4.0,3.11


In [120]:
# Keep the original activity dataframe unchanged

activity_clean = activity.rename(columns={
    "SEQN": "participant_id",
    "PAD790Q": "moderate_activity_frequency",
    "PAD790U": "moderate_activity_unit",
    "PAD800": "moderate_activity_minutes",
    "PAD810Q": "vigorous_activity_frequency",
    "PAD810U": "vigorous_activity_unit",
    "PAD820": "vigorous_activity_minutes",
    "PAD680": "sitting_minutes_per_day"
}).copy()

## Physical Activity Feature Engineering

The physical activity dataset reports moderate- and vigorous-intensity leisure-time activity using three pieces of information:

- **Frequency**: how often the participant performs the activity
- **Unit**: whether the frequency is reported per day, week, month, or year
- **Duration**: how many minutes the participant usually performs the activity each time

Because participants may report their activity using different time units, the raw values are not directly comparable.

Therefore, the activity information will be converted to a common **weekly scale**.

Two new variables will be created:

- **`moderate_minutes_per_week`**: estimated total minutes of moderate-intensity physical activity per week.
- **`vigorous_minutes_per_week`**: estimated total minutes of vigorous-intensity physical activity per week.

For example, if a participant reports moderate activity **3 times per week for 45 minutes each time**, the estimated weekly moderate activity is:

**3 × 45 = 135 minutes per week**

Participants who explicitly report no moderate or vigorous activity will be assigned **0 minutes per week** for that activity type. Other genuinely missing responses will remain missing and will be handled later during data cleaning.

In [121]:
unit_to_week = {
    b"D": 7,
    b"W": 1,
    b"M": 12 / 52,
    b"Y": 1 / 52
}
activity_clean["moderate_minutes_per_week"] = (
    activity_clean["moderate_activity_frequency"]
    * activity_clean["moderate_activity_unit"].map(unit_to_week)
    * activity_clean["moderate_activity_minutes"]
)

activity_clean["vigorous_minutes_per_week"] = (
    activity_clean["vigorous_activity_frequency"]
    * activity_clean["vigorous_activity_unit"].map(unit_to_week)
    * activity_clean["vigorous_activity_minutes"]
)
activity_clean

,participant_id,moderate_activity_frequency,moderate_activity_unit,moderate_activity_minutes,vigorous_activity_frequency,vigorous_activity_unit,vigorous_activity_minutes,sitting_minutes_per_day,moderate_minutes_per_week,vigorous_minutes_per_week
0,130378.0,3.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,360.0,135.0,135.0
1,130379.0,4.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,480.0,180.0,135.0
2,130380.0,1.000000e+00,b'W',20.0,5.397605e-79,b'',NaN,240.0,20.0,NaN
3,130384.0,5.397605e-79,b'',NaN,5.397605e-79,b'',NaN,60.0,NaN,NaN
4,130385.0,1.000000e+00,b'D',90.0,1.000000e+00,b'W',60.0,180.0,630.0,60.0
...,...,...,...,...,...,...,...,...,...,...
8148,142305.0,2.000000e+00,b'W',40.0,5.397605e-79,b'',NaN,480.0,80.0,NaN
8149,142307.0,3.000000e+00,b'W',15.0,5.397605e-79,b'',NaN,480.0,45.0,NaN
8150,142308.0,1.000000e+00,b'W',45.0,5.397605e-79,b'',NaN,600.0,45.0,NaN
8151,142309.0,2.000000e+00,b'D',15.0,5.397605e-79,b'',NaN,240.0,210.0,NaN


In [114]:
#Rename the Body columns
body = body.rename(columns={
    "SEQN": "participant_id",
    "BMDSTATS": "body_measure_status",

    "BMXWT": "weight_kg",
    "BMIWT": "weight_comment",

    "BMXRECUM": "recumbent_length_cm",
    "BMIRECUM": "recumbent_length_comment",

    "BMXHEAD": "head_circumference_cm",
    "BMIHEAD": "head_circumference_comment",

    "BMXHT": "height_cm",
    "BMIHT": "height_comment",

    "BMXBMI": "bmi",
    "BMDBMIC": "bmi_category_child_youth",

    "BMXLEG": "upper_leg_length_cm",
    "BMILEG": "upper_leg_length_comment",

    "BMXARML": "upper_arm_length_cm",
    "BMIARML": "upper_arm_length_comment",

    "BMXARMC": "arm_circumference_cm",
    "BMIARMC": "arm_circumference_comment",

    "BMXWAIST": "waist_circumference_cm",
    "BMIWAIST": "waist_circumference_comment",

    "BMXHIP": "hip_circumference_cm",
    "BMIHIP": "hip_circumference_comment"
})
#The columns I need for the analysis
body = body[
    [
        "participant_id",
        "weight_kg",
        "height_cm",
        "bmi",
        "waist_circumference_cm",
        "hip_circumference_cm"
    ]
]
body

,participant_id,weight_kg,height_cm,bmi,waist_circumference_cm,hip_circumference_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
3,130381.0,34.3,120.1,23.8,70.4,NaN
4,130382.0,13.6,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
8855,142306.0,25.3,128.0,15.4,57.7,NaN
8856,142307.0,NaN,143.8,NaN,NaN,NaN
8857,142308.0,79.3,173.3,26.4,98.4,97.7
8858,142309.0,81.9,179.1,25.5,96.0,103.3


In [115]:
#Rename the blood pressure columns
bp = bp.rename(columns={
    "SEQN": "participant_id",

    "BPAOARM": "measurement_arm",
    "BPAOCSZ": "cuff_size_code",

    "BPXOSY1": "systolic_bp_1",
    "BPXODI1": "diastolic_bp_1",

    "BPXOSY2": "systolic_bp_2",
    "BPXODI2": "diastolic_bp_2",

    "BPXOSY3": "systolic_bp_3",
    "BPXODI3": "diastolic_bp_3",

    "BPXOPLS1": "pulse_1",
    "BPXOPLS2": "pulse_2",
    "BPXOPLS3": "pulse_3"
})
#Keeping the columns that needed
bp = bp[
    [
        "participant_id",
        "systolic_bp_1",
        "diastolic_bp_1",
        "systolic_bp_2",
        "diastolic_bp_2",
        "systolic_bp_3",
        "diastolic_bp_3"
    ]
]
bp.head()

,participant_id,systolic_bp_1,diastolic_bp_1,systolic_bp_2,diastolic_bp_2,systolic_bp_3,diastolic_bp_3
0,130378.0,135.0,98.0,131.0,96.0,132.0,94.0
1,130379.0,121.0,84.0,117.0,76.0,113.0,76.0
2,130380.0,111.0,79.0,112.0,80.0,104.0,76.0
3,130386.0,110.0,72.0,120.0,74.0,115.0,75.0
4,130387.0,143.0,76.0,136.0,74.0,145.0,78.0


## Blood Pressure Feature Engineering

The blood pressure dataset contains three systolic and three diastolic blood pressure measurements for each participant.

- **Systolic blood pressure** is the pressure in the arteries when the heart contracts and pumps blood.
- **Diastolic blood pressure** is the pressure in the arteries when the heart relaxes between beats.

Because blood pressure was measured three times, a single average value is calculated for each participant.

Two new variables are created:

- **`mean_systolic_bp`**: average of the three systolic blood pressure measurements.
- **`mean_diastolic_bp`**: average of the three diastolic blood pressure measurements.

Using the average of repeated measurements provides one representative blood pressure value for each participant and makes the later analysis easier to interpret.

In [116]:
bp["average_upper_bp"] = bp[
    ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
].mean(axis=1)

bp["average_lower_bp"] = bp[
    ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]
].mean(axis=1)
bp

,participant_id,systolic_bp_1,diastolic_bp_1,systolic_bp_2,diastolic_bp_2,systolic_bp_3,diastolic_bp_3,average_upper_bp,average_lower_bp
0,130378.0,135.0,98.0,131.0,96.0,132.0,94.0,132.666667,96.000000
1,130379.0,121.0,84.0,117.0,76.0,113.0,76.0,117.000000,78.666667
2,130380.0,111.0,79.0,112.0,80.0,104.0,76.0,109.000000,78.333333
3,130386.0,110.0,72.0,120.0,74.0,115.0,75.0,115.000000,73.666667
4,130387.0,143.0,76.0,136.0,74.0,145.0,78.0,141.333333,76.000000
...,...,...,...,...,...,...,...,...,...
7796,142306.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7797,142307.0,127.0,75.0,132.0,73.0,131.0,72.0,130.000000,73.333333
7798,142308.0,106.0,65.0,106.0,69.0,112.0,74.0,108.000000,69.333333
7799,142309.0,127.0,81.0,125.0,82.0,128.0,81.0,126.666667,81.333333


**Rmoving NAs**

In [117]:
demo_clean = demo.dropna(
    subset=[
        "participant_id",
        "age_years",
        "sex",
        "education_level",
        "income_poverty_ratio",
        "exam_weight"
    ]
)
print("Rows before cleaning:", len(demo))
print("Rows after cleaning:", len(demo_clean))
print("Rows removed:", len(demo) - len(demo_clean))

demo_clean["sex"] = demo_clean["sex"].replace({
    1: "Male",
    2: "Female"
})
demo_clean.loc[
    demo_clean["exam_weight"] < 1e-50,
    "exam_weight"
] = 0
demo_clean["exam_weight"] = demo_clean["exam_weight"].round(2)
demo_clean["education_level"] = demo_clean["education_level"].replace({
    1: "Less than 9th grade",
    2: "9th-11th grade",
    3: "High school/GED",
    4: "Some college/AA degree",
    5: "College graduate or above"
})
demo_clean

Rows before cleaning: 11933
Rows after cleaning: 6488
Rows removed: 5445


,participant_id,age_years,sex,exam_weight,education_level,income_poverty_ratio
0,130378.0,43.0,Male,54374.46,College graduate or above,5.00
1,130379.0,66.0,Male,34084.72,College graduate or above,5.00
2,130380.0,44.0,Female,81196.28,High school/GED,1.41
6,130384.0,43.0,Male,0.00,9th-11th grade,0.63
7,130385.0,65.0,Female,0.00,High school/GED,5.00
...,...,...,...,...,...,...
11925,142303.0,69.0,Female,47087.58,High school/GED,0.98
11927,142305.0,76.0,Female,43483.41,Less than 9th grade,2.25
11930,142308.0,50.0,Male,44367.53,Some college/AA degree,1.95
11931,142309.0,40.0,Male,46249.36,Some college/AA degree,3.11


1.00  = family income is about equal to the poverty guideline
0.50  = income is about 50% of the poverty guideline
2.00  = income is about twice the poverty guideline
3.00  = income is about three times the poverty guideline
5.00  = income is at least five times the poverty guideline

In [118]:
demo_clean.loc[
    demo_clean["income_poverty_ratio"] < 1e-50,
    "income_poverty_ratio"
] = 0
demo_clean["poverty_group"] = pd.cut(
    demo_clean["income_poverty_ratio"],
    bins=[-0.01, 0.99, 1.29, 1.84, 3.49, 4.99, float("inf")],
    labels=[
        "Below poverty",
        "Around poverty",
        "Low income",
        "Middle income",
        "Higher income",
        "5x poverty or more"
    ]
)
demo_clean

,participant_id,age_years,sex,exam_weight,education_level,income_poverty_ratio,poverty_group
0,130378.0,43.0,Male,54374.46,College graduate or above,5.00,5x poverty or more
1,130379.0,66.0,Male,34084.72,College graduate or above,5.00,5x poverty or more
2,130380.0,44.0,Female,81196.28,High school/GED,1.41,Low income
6,130384.0,43.0,Male,0.00,9th-11th grade,0.63,Below poverty
7,130385.0,65.0,Female,0.00,High school/GED,5.00,5x poverty or more
...,...,...,...,...,...,...,...
11925,142303.0,69.0,Female,47087.58,High school/GED,0.98,Below poverty
11927,142305.0,76.0,Female,43483.41,Less than 9th grade,2.25,Middle income
11930,142308.0,50.0,Male,44367.53,Some college/AA degree,1.95,Middle income
11931,142309.0,40.0,Male,46249.36,Some college/AA degree,3.11,Middle income


In [119]:
demo_clean["age_group"] = pd.cut(
    demo_clean["age_years"],
    bins=[19, 39, 59, float("inf")],
    labels=[
        "Young adults",
        "Middle-aged adults",
        "Older adults"
    ]
)
demo_clean

,participant_id,age_years,sex,exam_weight,education_level,income_poverty_ratio,poverty_group,age_group
0,130378.0,43.0,Male,54374.46,College graduate or above,5.00,5x poverty or more,Middle-aged adults
1,130379.0,66.0,Male,34084.72,College graduate or above,5.00,5x poverty or more,Older adults
2,130380.0,44.0,Female,81196.28,High school/GED,1.41,Low income,Middle-aged adults
6,130384.0,43.0,Male,0.00,9th-11th grade,0.63,Below poverty,Middle-aged adults
7,130385.0,65.0,Female,0.00,High school/GED,5.00,5x poverty or more,Older adults
...,...,...,...,...,...,...,...,...
11925,142303.0,69.0,Female,47087.58,High school/GED,0.98,Below poverty,Older adults
11927,142305.0,76.0,Female,43483.41,Less than 9th grade,2.25,Middle income,Older adults
11930,142308.0,50.0,Male,44367.53,Some college/AA degree,1.95,Middle income,Middle-aged adults
11931,142309.0,40.0,Male,46249.36,Some college/AA degree,3.11,Middle income,Middle-aged adults


In [122]:

activity_clean.loc[
    activity_clean["moderate_activity_frequency"] < 1e-50,
    "moderate_activity_frequency"
] = 0

activity_clean.loc[
    activity_clean["vigorous_activity_frequency"] < 1e-50,
    "vigorous_activity_frequency"
] = 0
activity_clean[
    [
        "moderate_activity_frequency",
        "vigorous_activity_frequency"
    ]
].describe()
activity_clean

,participant_id,moderate_activity_frequency,moderate_activity_unit,moderate_activity_minutes,vigorous_activity_frequency,vigorous_activity_unit,vigorous_activity_minutes,sitting_minutes_per_day,moderate_minutes_per_week,vigorous_minutes_per_week
0,130378.0,3.0,b'W',45.0,3.0,b'W',45.0,360.0,135.0,135.0
1,130379.0,4.0,b'W',45.0,3.0,b'W',45.0,480.0,180.0,135.0
2,130380.0,1.0,b'W',20.0,0.0,b'',NaN,240.0,20.0,NaN
3,130384.0,0.0,b'',NaN,0.0,b'',NaN,60.0,NaN,NaN
4,130385.0,1.0,b'D',90.0,1.0,b'W',60.0,180.0,630.0,60.0
...,...,...,...,...,...,...,...,...,...,...
8148,142305.0,2.0,b'W',40.0,0.0,b'',NaN,480.0,80.0,NaN
8149,142307.0,3.0,b'W',15.0,0.0,b'',NaN,480.0,45.0,NaN
8150,142308.0,1.0,b'W',45.0,0.0,b'',NaN,600.0,45.0,NaN
8151,142309.0,2.0,b'D',15.0,0.0,b'',NaN,240.0,210.0,NaN


In [123]:
activity_clean.loc[
    activity_clean["moderate_activity_frequency"] == 0,
    "moderate_minutes_per_week"
] = 0

activity_clean.loc[
    activity_clean["vigorous_activity_frequency"] == 0,
    "vigorous_minutes_per_week"
] = 0
activity_clean

,participant_id,moderate_activity_frequency,moderate_activity_unit,moderate_activity_minutes,vigorous_activity_frequency,vigorous_activity_unit,vigorous_activity_minutes,sitting_minutes_per_day,moderate_minutes_per_week,vigorous_minutes_per_week
0,130378.0,3.0,b'W',45.0,3.0,b'W',45.0,360.0,135.0,135.0
1,130379.0,4.0,b'W',45.0,3.0,b'W',45.0,480.0,180.0,135.0
2,130380.0,1.0,b'W',20.0,0.0,b'',NaN,240.0,20.0,0.0
3,130384.0,0.0,b'',NaN,0.0,b'',NaN,60.0,0.0,0.0
4,130385.0,1.0,b'D',90.0,1.0,b'W',60.0,180.0,630.0,60.0
...,...,...,...,...,...,...,...,...,...,...
8148,142305.0,2.0,b'W',40.0,0.0,b'',NaN,480.0,80.0,0.0
8149,142307.0,3.0,b'W',15.0,0.0,b'',NaN,480.0,45.0,0.0
8150,142308.0,1.0,b'W',45.0,0.0,b'',NaN,600.0,45.0,0.0
8151,142309.0,2.0,b'D',15.0,0.0,b'',NaN,240.0,210.0,0.0


In [124]:
activity_clean = activity_clean.dropna(
    subset=[
        "moderate_minutes_per_week",
        "vigorous_minutes_per_week",
        "sitting_minutes_per_day"
    ]
).copy()
print("Rows before:", len(activity))
print("Rows after:", len(activity_clean))
print("Rows removed:", len(activity) - len(activity_clean))
activity_clean.loc[
    activity_clean["moderate_activity_frequency"] == 0,
    "moderate_activity_unit"
] = "No activity"

activity_clean.loc[
    activity_clean["vigorous_activity_frequency"] == 0,
    "vigorous_activity_unit"
] = "No activity"
unit_labels = {
    b"D": "Day",
    b"W": "Week",
    b"M": "Month",
    b"Y": "Year"
}

activity_clean["moderate_activity_unit"] = (
    activity_clean["moderate_activity_unit"].replace(unit_labels)
)

activity_clean["vigorous_activity_unit"] = (
    activity_clean["vigorous_activity_unit"].replace(unit_labels)
)

Rows before: 8153
Rows after: 8057
Rows removed: 96


In [125]:
activity_clean

,participant_id,moderate_activity_frequency,moderate_activity_unit,moderate_activity_minutes,vigorous_activity_frequency,vigorous_activity_unit,vigorous_activity_minutes,sitting_minutes_per_day,moderate_minutes_per_week,vigorous_minutes_per_week
0,130378.0,3.0,Week,45.0,3.0,Week,45.0,360.0,135.0,135.0
1,130379.0,4.0,Week,45.0,3.0,Week,45.0,480.0,180.0,135.0
2,130380.0,1.0,Week,20.0,0.0,No activity,NaN,240.0,20.0,0.0
3,130384.0,0.0,No activity,NaN,0.0,No activity,NaN,60.0,0.0,0.0
4,130385.0,1.0,Day,90.0,1.0,Week,60.0,180.0,630.0,60.0
...,...,...,...,...,...,...,...,...,...,...
8148,142305.0,2.0,Week,40.0,0.0,No activity,NaN,480.0,80.0,0.0
8149,142307.0,3.0,Week,15.0,0.0,No activity,NaN,480.0,45.0,0.0
8150,142308.0,1.0,Week,45.0,0.0,No activity,NaN,600.0,45.0,0.0
8151,142309.0,2.0,Day,15.0,0.0,No activity,NaN,240.0,210.0,0.0


In [126]:
body_clean = body.copy()
body_clean=body_clean.dropna()
print("Rows before:", len(body))
print("Rows after:", len(body_clean))
print("Rows removed:", len(body) - len(body_clean))
body_clean

Rows before: 8860
Rows after: 6749
Rows removed: 2111


,participant_id,weight_kg,height_cm,bmi,waist_circumference_cm,hip_circumference_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
5,130386.0,90.6,173.3,30.2,106.1,110.6
6,130387.0,103.5,155.9,42.6,122.0,148.9
...,...,...,...,...,...,...
8853,142304.0,48.9,162.4,18.5,69.6,81.4
8854,142305.0,60.4,151.4,26.4,89.0,96.4
8857,142308.0,79.3,173.3,26.4,98.4,97.7
8858,142309.0,81.9,179.1,25.5,96.0,103.3


In [127]:
bp["average_upper_bp"] = (
    bp[["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]]
    .mean(axis=1)
    .round(2)
)

bp["average_lower_bp"] = (
    bp[["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]]
    .mean(axis=1)
    .round(2)
)
bp_clean = bp[
    [
        "participant_id",
        "average_upper_bp",
        "average_lower_bp"
    ]
].dropna().copy()

In [128]:
bp_clean.describe()

,participant_id,average_upper_bp,average_lower_bp
count,7518.000000,7518.000000,7518.000000
mean,136338.014099,119.094465,72.207223
std,3453.214803,18.151734,11.471162
min,130378.000000,70.000000,34.000000
25%,133316.250000,106.330000,64.000000
50%,136362.500000,116.330000,71.670000
75%,139316.500000,129.000000,79.330000
max,142310.000000,232.330000,139.000000


In [129]:
demo_clean
body_clean
activity_clean
bp_clean
body_clean

,participant_id,weight_kg,height_cm,bmi,waist_circumference_cm,hip_circumference_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
5,130386.0,90.6,173.3,30.2,106.1,110.6
6,130387.0,103.5,155.9,42.6,122.0,148.9
...,...,...,...,...,...,...
8853,142304.0,48.9,162.4,18.5,69.6,81.4
8854,142305.0,60.4,151.4,26.4,89.0,96.4
8857,142308.0,79.3,173.3,26.4,98.4,97.7
8858,142309.0,81.9,179.1,25.5,96.0,103.3


---
## 3. Design your tables

Decide what your tables are and how they relate before you write any code. Draw the ERD (Excalidraw, draw.io), export it as an image, commit it, and link it from your README.

Three tables minimum, with real primary and foreign keys.

In [130]:
participants = demo[["participant_id"]].dropna().drop_duplicates().copy()

participants["participant_id"] = participants["participant_id"].astype(int)
participants

,participant_id
0,130378
1,130379
2,130380
3,130381
4,130382
...,...
11928,142306
11929,142307
11930,142308
11931,142309


## 3. Design Your Tables

The cleaned NHANES data are organized into four related tables.  
The `demographics` table is the main table, and the other three tables are connected to it using `participant_id`.

### Tables

**`demographics`**
- `participant_id` — Primary Key
- `age_years`
- `sex`
- `exam_weight`
- `education_level`
- `income_poverty_ratio`
- `age_group`
- `poverty_group`

**`physical_activity`**
- `participant_id` — Primary Key and Foreign Key
- `moderate_activity_frequency`
- `moderate_activity_unit`
- `moderate_activity_minutes`
- `vigorous_activity_frequency`
- `vigorous_activity_unit`
- `vigorous_activity_minutes`
- `sitting_minutes_per_day`
- `moderate_minutes_per_week`
- `vigorous_minutes_per_week`

**`body_measurements`**
- `participant_id` — Primary Key and Foreign Key
- `weight_kg`
- `height_cm`
- `bmi`
- `waist_circumference_cm`
- `hip_circumference_cm`

**`blood_pressure`**
- `participant_id` — Primary Key and Foreign Key
- `average_upper_bp`
- `average_lower_bp`

### Relationships

All tables are linked through `participant_id`.

```text
demographics
    |
    |-- physical_activity
    |
    |-- body_measurements
    |
    |-- blood_pressure
```

Each participant appears once in each table, and `participant_id` in the three measurement tables refers to the same participant in `demographics`.

This structure allows the datasets to be joined later in SQL for analysis while keeping the database organized and relational.

![NHANES ERD](../ERD1.png)

---
## 4. Build the tables

Split the cleaned data into the tables you designed.

In [ ]:
participants
demo_clean
activity_clean
body_clean
bp_clean

---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

In [131]:
parent_ids = set(participants["participant_id"])

tables = {
    "demo_clean": demo_clean,
    "activity_clean": activity_clean,
    "body_clean": body_clean,
    "bp_clean": bp_clean
}

for name, df in tables.items():
    dangling = df.loc[
        ~df["participant_id"].isin(parent_ids),
        "participant_id"
    ]

    print(f"{name}: {len(dangling)} dangling foreign keys")

demo_clean: 0 dangling foreign keys
activity_clean: 0 dangling foreign keys
body_clean: 0 dangling foreign keys
bp_clean: 0 dangling foreign keys


---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

In [132]:
participants.to_csv(
    "../data/clean/participants.csv",
    index=False
)

demo_clean.to_csv(
    "../data/clean/demographics.csv",
    index=False
)

activity_clean.to_csv(
    "../data/clean/physical_activity.csv",
    index=False
)

body_clean.to_csv(
    "../data/clean/body_measurements.csv",
    index=False
)

bp_clean.to_csv(
    "../data/clean/blood_pressure.csv",
    index=False
)

In [133]:
import sqlite3

In [142]:
conn.close()
import os

# Delete the old database so we can rebuild it with the corrected data
if os.path.exists("nhanes.db"):
    os.remove("nhanes.db")

In [145]:

conn = sqlite3.connect("../data/nhanes.db")
conn.execute("PRAGMA foreign_keys = ON")

In [146]:
# ---------------------------------------------------
# 1. Read the SQL schema file
# ---------------------------------------------------

# Open the schema.sql file and read all CREATE TABLE statements
with open("sql/schema.sql", "r") as file:
    schema = file.read()

# Execute all SQL statements from the schema file
# This creates the empty database tables
conn.executescript(schema)

In [147]:
# ---------------------------------------------------
# 2. Load the parent table first
# ---------------------------------------------------

# The participants table is the parent table.
# It must be loaded first because all other tables
# reference participant_id from this table.

participants.to_sql(
    "participants",       # name of the SQL table
    conn,                 # database connection
    if_exists="append",   # add data to the existing table
    index=False           # do not save the pandas index
)

11933

In [148]:
# ---------------------------------------------------
# 3. Load the child tables
# ---------------------------------------------------

# Load the cleaned demographics table
demo_clean.to_sql(
    "demographics",
    conn,
    if_exists="append",
    index=False
)

# Load the cleaned physical activity table
activity_clean.to_sql(
    "physical_activity",
    conn,
    if_exists="append",
    index=False
)

# Load the cleaned body measurements table
body_clean.to_sql(
    "body_measurements",
    conn,
    if_exists="append",
    index=False
)

# Load the cleaned blood pressure table
bp_clean.to_sql(
    "blood_pressure",
    conn,
    if_exists="append",
    index=False
)

7518

In [149]:
# ---------------------------------------------------
# 4. Check how many rows were loaded into each table
# ---------------------------------------------------

tables = [
    "participants",
    "demographics",
    "physical_activity",
    "body_measurements",
    "blood_pressure"
]

for table in tables:

    # Count all rows in the current SQL table
    result = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {result} rows")

participants: 11933 rows
demographics: 6488 rows
physical_activity: 8057 rows
body_measurements: 6749 rows
blood_pressure: 7518 rows


In [150]:
# ---------------------------------------------------
# 5. Check for foreign-key violations
# ---------------------------------------------------

# SQLite checks whether every foreign key points
# to an existing participant in the participants table

foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check"
).fetchall()

print(foreign_key_errors)

[]


No foreign-key violations were found.

---
## 7. Verify

Row counts per table, and one join that returns the rows you expect. Then open `data/project.db` in DB Browser and look at it.